# OceanCleanup System 03 — Optimization Example

This notebook adapts the multi-objective optimization workflow from the lecture "System design principles" from 15.09.2026 and the example workbook: `EXAMPLE_Artificial_Reef.ipynb` to the OceanCleanup System 03 offshore plastic-collection barrier. It follows the same structure:

- Define design variables and bounds
- Specify multiple objective functions and preference mappings
- Build constraints
- Run a Genetic Algorithm (GA) using two (potentially three) aggregation paradigms
- Visualize preference functions and optimization results

## Problem

System 03 is a passive, offshore plastic-collection system: a long U-shaped floating barrier towed slowly through a garbage patch by two vessels, funneling floating debris toward a retention zone at the apex. Unlike a fixed coastal structure, its design is a trade-off between capturing as much plastic as possible and avoiding harm to marine life, all while remaining operable by a small crew at reasonable cost.

Design choices — how deep its skirt hangs below the surface, how far apart the two towing vessels hold the ends, how fast the system is towed, how fine its screen mesh is, and how many systems are deployed — all affect capture rate, bycatch risk, fishing-ground blockage, and cost. The barrier length itself is treated as fixed **(maybe this need so be adapted in the future when we want to experiment with the vessel spacing )**.

**Note:** Values in the following implementation are placeholders and not researched values.

## Importing Required Packages


In [1]:
# Import libraries
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import pchip_interpolate
from scipy.optimize import minimize

# Define default plotting parameters
plt.rcParams['font.size'] = '10'
plt.rcParams['savefig.dpi'] = 300

# Import local module for genetic algorithm
from genetic_algorithm_pfm import GeneticAlgorithm

## Design Variables and Bounds

The System 03 design is parameterized using five design variables (four continuous, one integer). They follow from the stakeholder objectives shown below: each variable influences at least one objective, and most influence several, which is what makes this a trade-off problem. The variables are numbered from left to right as in the scheme.

![Stakeholder objectives and design variables](../Our%20project/Stakeholder_Objectives_and_optimisation/stakeholder_objective_variable_scheme.png)

| Variable | Description | Unit | Type |
|----------|-------------|------|------|
| `x1` | Skirt depth below waterline | m | Continuous |
| `x2` | Distance between support vessels (vessel spacing) | m | Continuous |
| `x3` | Number of systems deployed | – | Integer |
| `x4` | Towing speed | m/s | Continuous |
| `x5` | Mesh / screen size | mm | Continuous |

Bounds and values below are placeholder estimates — replace with values we can justify from project research:

- `x1` skirt depth: a deeper skirt catches more submerged plastic but adds material, drag, and contact with animals in the water column.
- `x2` vessel spacing: a wider mouth captures more plastic but needs more towing force and closes more sea area, meaning fishing area to fishers.
- `x3` number of systems: more systems remove more plastic in total, but raise the total cost, the area closed to fishers, and the ecological contact. It must be a whole number, so it is marked `'int'` in `var_type_mixed`.
- `x4` towing speed: a higher speed captures more per hour but raises drag and fuel cost, gives animals less time to avoid the barrier, and lets small fragments escape.
- `x5` mesh size: a finer mesh retains more microplastic but increases fouling and the risk of trapping small marine life.

`X_irl` holds a rough reference design of today's situation (one System 03, so `x3 = 1`) for later comparison against the optimized solutions — update it once you have better sources.

In [2]:
# Define the names of variables for later use in plotting and analysis
'''
All of those values are place holders: put in own values.
'''
design_variables = (
    ('x1', 'Skirt depth below waterline',      'm'),
    ('x2', 'Distance between support vessels', 'm'),
    ('x3', 'Number of systems deployed',       '-'),
    ('x4', 'Towing speed',                     'm/s'),
    ('x5', 'Mesh / screen size',               'mm')
)

# Fixed system parameter - NOT a design variable (use it in the objective functions where needed)
barrier_length = 2500   # m, total length of the barrier (about 2.5 km according to The Ocean Cleanup website -> 2024)

# set bounds for all variables
b1 = [1, 6]          # x1 skirt depth
b2 = [200, 2000]     # x2 distance between vessels (must stay below barrier_length)
b3 = [1, 10]         # x3 number of systems
b4 = [0.2, 1.0]      # x4 towing speed
b5 = [1, 50]         # x5 mesh size
bounds = [b1, b2, b3, b4, b5]

# type of each variable: 'real' = continuous, 'int' = whole number (used in the GA options later)
var_type_mixed = ['real', 'real', 'int', 'real', 'real']

X_irl = [4, 1200, 1, 0.5, 10] # real world values --> place holders, put in later
plot_irl = True  # Can be True or False --> plot the IRL point or not

## Constraints

Constraints ensure that all solutions remain physically feasible. For this artificial reef model, one physical constraint is considered: **the reef has a minimum distance from the beach**. This constraint, `constraint_1`, is a simple geometric relation corresponding to the distance `x5` shown in the figure above. A constraint function may use as many or as few design variables as necessary.

Each *constraint function* must be expressed as an algebraic expression of the form:

$$g(\mathbf{x}) \leq 0 \quad \text{(feasible)}$$

> **Important:** The constraint function should return only the value of $g(\mathbf{x})$ — do *not* encode the inequality inside the function. The inequality type (`'ineq'`) is specified separately when registering the constraint in `cons`, and the optimiser handles the comparison.

The constraints are registered as follows:

```python
cons = [['ineq', constraint_1], ['ineq', constraint_2]]

In [3]:
def constraint_1(variables):
    """Cable Tension:

    :return: 1-D array (length n) with constraint values; arr>0 means violation for 'ineq' constraints.
    """
    x1 = variables[:, 0]
    x2 = variables[:, 1]
    x3 = variables[:, 2]
    x4 = variables[:, 3]
    x5 = variables[:, 4]
    max_tension = 30      # maximum admisible tension of the cable [kN]
    Cd = 1 #Hydrodynamic drag coefficient.
    
    return Cd * (barrier_length * x1) * x4 ** 2 * (x2 / barrier_length) * (1 / x5) # If the force exceeds the max tension it can broke.

def constraint_2(variables):
    """Maximum volume:

    :return: 1-D array (length n) with constraint values; arr>0 means violation for 'ineq' constraints.
    """
    x1 = variables[:, 0]
    x2 = variables[:, 1]
    x3 = variables[:, 2]
    x4 = variables[:, 3]
    x5 = variables[:, 4]

    max_volume = 1540 #[m3], maximum theoretical volumetric capacity.
    Kvol = 0.38 #Geometric scaling constant that translates the bounding box of the span into the actual pool volume.
    
    return max_volume - (Kvol * x1 * x2 * ((barrier_length**2) * (x2 ** 2)) ** (1/2)) # < 0  # Max volume: if the result is negative, it means the capacity of that the waste bag can take is exceeded.

def constraint_3(variables):
    """Vessel autonomy constraint

    :return: 1-D array (length n) with constraint values; arr>0 means violation for 'ineq' constraints.
    """
    x1 = variables[:, 0]
    x2 = variables[:, 1]
    x3 = variables[:, 2]
    x4 = variables[:, 3]
    x5 = variables[:, 4]

    K_base = 1740.74 #[L/(system * hour * (m/s)**3)]
    K_weight = 668.80 #[L/(system * hour * (m/s)**2)]

    Fuel_max = 800 #L
    
    return - ((K_base * x3 * x4 ** 3) + (K_weight * 0.50 * (x4 ** 2) * x3)) + Fuel_max # < 0  # If the result is negative, we don't have enough fuel to the system.

cons = [['ineq', constraint_1], ['ineq', constraint_2], ['ineq', constraint_3]]